# PySpark 4: windows and nested data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/spark/pyspark-04-windows-and-nested-data.ipynb)

Window functions compute values across related rows without collapsing them. This module also covers everyday manipulation (dates, time zones, nulls, flags, deduplication) and flattening nested JSON and arrays.

**You will learn to:**

- Rank and deduplicate rows with `row_number` over a window
- Compute running totals and comparisons to the previous row
- Convert time zones and derive dates safely
- Parse nested JSON and explode arrays without losing rows

**Before you start:** PySpark 1 to 3.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

## How do window functions work in PySpark?

Window functions calculate values across related rows without collapsing the result like `groupBy` does.

In [ ]:
from pyspark.sql import Window, functions as F

w = Window.partitionBy("customer_id").orderBy(F.col("order_ts").desc())

latest_order = (
    orders
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
latest_order.select("customer_id", "order_id", "status", "order_ts", "amount").show()

# the window kept one row per customer instead of collapsing the frame like groupBy
assert latest_order.count() == 6

Common window use cases:

+ Deduplicating CDC records by latest sequence number
+ Finding the latest row per business key
+ Calculating rolling metrics
+ Ranking products or customers within a segment

CDC example: several change rows per order, one replayed with the same LSN. Order by the
source sequence, then by ingestion time as a deterministic tie-break:

In [ ]:
from pyspark.sql import Window

orders_cdc = spark.createDataFrame(
    [
        (101, "placed",    1, datetime(2026, 1, 5, 9, 0)),
        (101, "paid",      2, datetime(2026, 1, 5, 9, 5)),
        (101, "shipped",   3, datetime(2026, 1, 6, 8, 0)),
        (102, "placed",    1, datetime(2026, 1, 5, 10, 0)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)),
        (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9)),  # replayed change, later ingest
    ],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)

w = Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc(), F.col("ingested_at").desc())

deduped_cdc = orders_cdc.withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn")
deduped_cdc.orderBy("order_id").show()

latest = {r["order_id"]: (r["status"], r["ingested_at"].minute) for r in deduped_cdc.collect()}
assert latest == {101: ("shipped", 0), 102: ("cancelled", 9)}

How it works under the hood. A window with `partitionBy` plans as `Exchange hashpartitioning(key)` followed by `Sort` inside each partition, then the window expression runs in one pass. Window expressions without `partitionBy` send every row to a single partition, which turns a distributed job into one task. `row_number` needs the sort; `sum().over(rowsBetween(...))` reuses it.

In [ ]:
from pyspark.sql import Window, functions as F

seven_days = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("order_ts").cast("long"))
    .rangeBetween(-7 * 86400, 0)
)

rolling = (
    orders
    .withColumn("spend_7d", F.sum("amount").over(seven_days))
    .orderBy("customer_id", "order_ts")
)
rolling.select("customer_id", "order_ts", "amount", "spend_7d").show()

# order 5 (placed, 55.25) sits inside the trailing week of order 4 (paid, 260.00)
row = rolling.where("order_id = 5").first()
assert float(row["spend_7d"]) == 315.25

Worked example. A 500 GB events table deduplicated with `row_number` over `partitionBy(order_id).orderBy(ts desc)` shuffles the whole table once, even if most keys are unique and there is nothing to deduplicate.

Common mistake: omitting a deterministic tie-break in `orderBy`, so the "latest" row changes between runs. Add the source sequence number or another column that breaks ties the same way every time.

## What are common PySpark data manipulation examples?

Most PySpark pipeline work is ordinary data manipulation: parsing timestamps, deriving dates, handling nulls, standardizing strings, splitting arrays, and creating business flags. Prefer `pyspark.sql.functions` because those expressions stay inside Spark's optimizer.

**Dates and timestamps.**

In [ ]:
from pyspark.sql import functions as F

orders_with_dates = (
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("order_month", F.date_trunc("month", "order_ts"))
    .withColumn("order_year", F.year("order_ts"))
    .withColumn("order_week", F.weekofyear("order_ts"))
    .withColumn("ship_by_date", F.date_add("order_date", 2))
    .withColumn("days_since_order", F.datediff(F.current_date(), "order_date"))
)
(
    orders_with_dates
    .orderBy("order_id")
    .select("order_id", "order_ts", "order_date", "ship_by_date", "days_since_order")
    .show()
)

# `ship_by_date` is `order_date` plus exactly two days, for every order
assert orders_with_dates.where(F.datediff("ship_by_date", "order_date") != 2).count() == 0

**Time zone conversion.** Common when sources emit UTC but reporting is local:

In [ ]:
from pyspark.sql import functions as F

localized = (
    orders
    .withColumn(
        "order_ts_chicago",
        F.from_utc_timestamp("order_ts", "America/Chicago"),
    )
    .withColumn("order_date_chicago", F.to_date("order_ts_chicago"))
    .orderBy("order_id")
)
localized.select("order_id", "order_ts", "order_ts_chicago", "order_date_chicago").show()

# order 12 lands at 00:30 UTC: in Chicago it is still January 13, in the evening
row = localized.where("order_id = 12").first()
assert row["order_ts_chicago"].strftime("%Y-%m-%d %H:%M") == "2026-01-13 18:30"

**Null handling and default values.**

In [ ]:
from pyspark.sql import functions as F

# customer 4's region never arrived; give the gap a default instead of a silent null
customers_filled = (
    customers
    .withColumn("region", F.coalesce("region", F.lit("unknown")))
)
customers_filled.orderBy("customer_id").select("customer_id", "name", "region").show()

row = customers_filled.where("customer_id = 4").first()
assert row["region"] == "unknown"
assert customers_filled.where(F.col("region").isNull()).count() == 0

**Type casting and safe derived columns.**

In [ ]:
typed_items = (
    order_items
    .withColumn("line_amount", (F.col("qty") * F.col("unit_price")).cast("decimal(12,2)"))
    .withColumn("unit_check", F.when(F.col("qty") > 0, F.col("line_amount") / F.col("qty")))
    .withColumn("bad_cast", F.expr("try_cast('N/A' AS INT)"))  # try_cast returns NULL instead of raising
)
typed_items.select("order_id", "sku", "qty", "line_amount", "unit_check", "bad_cast").show(5)

assert typed_items.where(F.col("unit_check") != F.col("unit_price")).count() == 0
assert typed_items.where(F.col("bad_cast").isNotNull()).count() == 0

A cast that silently returns null is a data-quality bug waiting to happen. Spark 4 turns ANSI
mode on by default, so a plain `cast` raises on malformed input. `try_cast` returns null instead, and that null
needs the same check as any other: count it after the cast.

**Conditional business flags.**

In [ ]:
from pyspark.sql import functions as F

classified = (
    orders
    .withColumn(
        "order_size",
        F.when(F.col("amount") >= 1000, "enterprise")
         .when(F.col("amount") >= 100, "mid_market")
         .otherwise("small"),
    )
    .withColumn(
        "is_completed",
        F.col("status").isin("paid", "shipped"),
    )
)
classified.orderBy("order_id").select("order_id", "amount", "status", "order_size", "is_completed").show()

# one wholesale order (1,180.00), five mid-market, six small
assert classified.where("order_size = 'enterprise'").count() == 1
assert classified.where("order_size = 'mid_market'").count() == 5

**Deduplicating records.**

In [ ]:
# the no-argument form compares entire rows, so exactly the retried copy dies
deduped_events = events.dropDuplicates()

assert events.count() == 13 and deduped_events.count() == 12
deduped_events.orderBy("user_id", "ts").show()

How it works under the hood. Every `withColumn` adds a projection node, and each `dropDuplicates` or window adds a wide node that exchanges rows by key. Catalyst later fuses the narrow projections into one, but a dedupe still moves the whole dataset across the network. `dropDuplicates(["order_id"])` repartitions by `order_id`; the no-argument form compares entire rows.

In [ ]:
from pyspark.sql import functions as F

order_summary = (
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("customer_id", "order_date")
    .agg(
        F.count("*").alias("orders"),
        F.sum("amount").alias("revenue"),
        F.max("amount").alias("largest_order"),
    )
    .withColumn("revenue_per_order", F.round(F.col("revenue") / F.col("orders"), 2))
    .orderBy("customer_id", "order_date")
)
order_summary.show()

# regrouping the ledger must not move money: bucketed revenue equals ledger revenue
assert order_summary.agg(F.sum("revenue")).first()[0] == orders.agg(F.sum("amount")).first()[0]

Worked example. Deduplicating a 1 TB orders table with `dropDuplicates(["order_id"])` shuffles roughly 1 TB by that key and keeps one arbitrary row per key.

Common mistake: expecting `dropDuplicates` to keep the newest record. It has no ordering, so the survivor is non-deterministic; for CDC use `row_number()` over a window ordered by a source sequence and a tie-break.

## How do you handle nested JSON and arrays in PySpark?

Nested data is common in event, clickstream, API, and CDC payloads. Use struct and array functions instead of converting rows to Python dictionaries.

In [ ]:
nested_dir = os.path.join(WORK_DIR, "raw", "events")
os.makedirs(nested_dir, exist_ok=True)
nested_rows = [
    {"event_id": "e1", "user": {"id": 1}, "device": {"os": "ios"},
     "items": [{"sku": "SKU-4001", "quantity": 2}, {"sku": "SKU-4007", "quantity": 1}],
     "payload_json": '{"source": "email", "campaign": {"id": "c-9", "name": "winter"}}'},
    {"event_id": "e2", "user": {"id": 2}, "device": {"os": "android"},
     "items": [{"sku": "SKU-4002", "quantity": 1}],
     "payload_json": '{"source": "ads", "campaign": {"id": "c-3", "name": "search"}}'},
    {"event_id": "e3", "user": {"id": 3}, "device": {"os": "web"}, "items": None,
     "payload_json": '{"source": "direct", "campaign": null}'},
]
with open(os.path.join(nested_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in nested_rows))

raw_events = spark.read.schema(
    "event_id string, user struct<id:int>, device struct<os:string>, "
    "items array<struct<sku:string,quantity:int>>, payload_json string"
).json(nested_dir)

flattened = raw_events.select(
    "event_id",
    F.col("user.id").alias("user_id"),
    F.col("device.os").alias("device_os"),
    F.explode_outer("items").alias("item"),
)
items = flattened.select(
    "event_id", "user_id", F.col("item.sku").alias("sku"), F.col("item.quantity").alias("quantity")
)
items.orderBy("event_id", "sku").show()

# explode drops e3 (null array); explode_outer keeps it with null item columns
assert raw_events.select(F.explode("items")).count() == 3
assert items.count() == 4

If a payload column contains JSON strings, parse it with `from_json`:

In [ ]:
payload_schema = "source string, campaign struct<id:string,name:string>"

parsed = (
    raw_events
    .withColumn("payload", F.from_json("payload_json", payload_schema))
    .select("event_id", F.col("payload.source").alias("source"), F.col("payload.campaign.id").alias("campaign_id"))
)
parsed.orderBy("event_id").show()
assert {r["event_id"]: r["campaign_id"] for r in parsed.collect()} == {"e1": "c-9", "e2": "c-3", "e3": None}

Interview point: `explode` can multiply rows. Always know the expected array size and watch for unexpected row explosions.

How it works under the hood. Parquet and JSON readers expose nested objects as `struct` and `array` columns, and `F.col("user.id")` resolves the field at plan time with no data conversion. `explode` is a `Generate` operator that expands one row into many inside its partition; it is narrow, but it multiplies the row count, so every later shuffle and write processes more data.

Worked example. An events file with an average array length of 4 and 100 million events becomes 400 million rows after `explode`, so a downstream join or write handles four times the rows and roughly four times the bytes.

Common mistake: choosing `explode` when `explode_outer` is correct. Rows with a null or empty array simply disappear, which is a classic silent correctness bug in event pipelines.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- `row_number` over a partition keeps the latest record per key, which is the usual CDC dedup.
- `explode` multiplies rows. Count rows before and after it.
- Parse payload strings with an explicit schema so a missing field becomes null, not an error.

**Next:** [PySpark 5: writing, pipelines and quality](pyspark-05-writing-pipelines-and-quality.ipynb)